<div style="float: block; text-align: center; line-height: 1.7em">
    <span style="font-size: 2em; font-weight: bold"> Fatigue-Sleepiness in Irregular Workloads for Pilots </span><br>
    <span style="font-size: 1.5em; font-weight: bold"> Statistical Modelling - Perceived Severe Fatigue </span><br>
</div>

----

---

# 1. Loading Libraries and Packages

In [1]:
import os                                            # system library
import pandas as pd                                  # library to handle databases
import numpy as np                                   # library to handle arrays and matrices
import matplotlib.pyplot as plt                      # graphical package
import seaborn as sns                                # beautification graphical package
from IPython.display import display, HTML

os.environ['R_HOME'] = "C:/PROGRA~1/R/R-44~1.2"      # Necessary to configure R_HOME path
from pymer4.models import Lmer                       # Package for statistical modelling

import rpy2.robjects as ro                           # Forcing to UTF-8 locale
ro.r('Sys.setlocale("LC_ALL", "English_United States.UTF-8")')

from sklearn.metrics import roc_auc_score, roc_curve, classification_report, accuracy_score, confusion_matrix, auc
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, average_precision_score

----

# 2. Reading Pre-Processed Data

In [2]:
file = os.path.join('data','fat_wrk_mdl_train.csv')
try:
    df = pd.read_csv(file)
    print('-----------------------------------------------')
    print(f"\033[93mSuccess Data file read!\033[0m")
    print('-----------------------------------------------')
    # display(df.head(3))
except Exception as e:
    print('-----------------------------------------------')
    print(f"\033[91mAn error occurred:\033[0m {e}")
    print('-----------------------------------------------')

-----------------------------------------------
Success Data file read!
-----------------------------------------------


----

# 3. Verifying for Null rows

In [3]:
df['Sleep_quality'] = pd.to_numeric(df['Sleep_quality'], errors = 'coerce')
wsm = df.isna().sum()
display(pd.DataFrame(wsm).T)

,Id,TPFS_fat,duty_moment,DMod_fat,PDMod_fat,Sleep_quality,time_awake,duty_length,age,spsd
0,0,0,0,0,0,24,4,4,0,0


* As we can see above for covariates time_awake and sleep duration there are some null rows, so these rows containing null values will be taken away.

In [4]:
df = df.dropna()

wsm = df.isna().sum()
display(pd.DataFrame(wsm).T)

,Id,TPFS_fat,duty_moment,DMod_fat,PDMod_fat,Sleep_quality,time_awake,duty_length,age,spsd
0,0,0,0,0,0,0,0,0,0,0


* Now as we can see, there is no longer null rows!

---

# 4. Scalling some covariates

To avoid scalling bias in the model, we will rescalling some continuous covariates.

In [9]:
df[['Sleep_quality','time_awake','duty_length']].describe().T

,count,mean,std,min,25%,50%,75%,max
Sleep_quality,1026.0,7.321637,1.696441,2.000000,6.0,8.000000,9.000000,10.000000
time_awake,1026.0,3.872498,3.510890,0.250000,1.5,2.375000,5.016667,17.400000
duty_length,1026.0,7.291131,2.328124,1.683333,6.0,7.466667,8.850000,11.583333


Above is shown some descriptive statistics of the ordinal and continuous covariates, based on the covariate statistics we propose the max-min standardization.

In [10]:
df['Sleep_quality'] = df['Sleep_quality']/(df['Sleep_quality'].max()-df['Sleep_quality'].min())
df['time_awake'] = df['time_awake']/(df['time_awake'].max()-df['time_awake'].min())
df['duty_length'] = df['duty_length']/(df['duty_length'].max()-df['duty_length'].min())

---

# 5. Preliminary Regression Analysis

Once we have a repeated measure design or a panel like data, we choose a mixed logistic model given by:

<p style="text-align: center;"> $g(x_{ij},\beta_{0i},\beta_1, \beta_s) = \beta_{0i}+\beta_{1i}x_1 + x_{ij}^T\beta_s$  (level 1) </p>
<p style="text-align: center;"> $\beta_{0i} = \beta_0 + \alpha_i$  (level 1, random intercept) </p>

where $\alpha_i \sim N(0,\sigma_\alpha^2)$ .

with $\alpha_i$ representing the random intercept due to participant variability and $\beta_0$ the intercept or baselines.

The logistic link function os given by:

<p style="text-align: center;"> $$g(x_{ij},\beta_{0i},\beta_1, \beta_s) = ln\left[\frac{\pi(x_{ij},\beta_{0i},\beta_1, \beta_s)}{1-\pi(x_{ij},\beta_{0i},\beta_1, \beta_s)}\right]$$ </p>

with, 

<p style="text-align: center;"> $$\pi(x_{ij},\beta_{0i},\beta_1, \beta_s) = \frac{e^{\beta_{0i}+\beta_{1i}x_1 + x_{ij}^T\beta_s}}{1+e^{\beta_{0i}+\beta_{1i}x_1 + x_{ij}^T\beta_s}}$$</p>


## 5.1. Function to codify the covariates in pymer standards

In [11]:
def get_formula(covariates, target, prnt = False):
    formula = f'{target}~'
    for cvt in covariates:
        formula += cvt+'+'
    formula = formula[:-1]
    if prnt:
        print(f'formula:\n{formula}')
    return formula

## 5.2. First Iteration

In [13]:
df.columns

Index(['Id', 'TPFS_fat', 'duty_moment', 'DMod_fat', 'PDMod_fat',
       'Sleep_quality', 'time_awake', 'duty_length', 'age', 'spsd'],
      dtype='object')

In [15]:
#covariates = ['TPFS_fat','duty_moment','DMod_fat','PDMod_fat','Sleep_quality','time_awake','duty_length','age']
covariates = ['duty_moment','DMod_fat','Sleep_quality','time_awake','duty_length','age']

formula = get_formula(covariates, 'spsd', prnt=False)
ff = formula + '+(1|Id)'

print('----------------------------------------------------------------------------------------------------------')
print(f'\033[91mFormula: {ff}\033[0m')
print('----------------------------------------------------------------------------------------------------------')


model_LME = Lmer(ff, data = df, family = 'binomial')
mdf = model_LME.fit()

mdf.head(100)

----------------------------------------------------------------------------------------------------------
Formula: spsd~duty_moment+DMod_fat+Sleep_quality+time_awake+duty_length+age+(1|Id)
----------------------------------------------------------------------------------------------------------
Model failed to converge with max|grad| = 0.0292416 (tol = 0.002, component 1) 

Linear mixed model fit by maximum likelihood  ['lmerMod']
Formula: spsd~duty_moment+DMod_fat+Sleep_quality+time_awake+duty_length+age+(1|Id)

Family: binomial	 Inference: parametric

Number of observations: 1026	 Groups: {'Id': 44.0}

Log-likelihood: -181.070 	 AIC: 382.140

Random effects:

           Name    Var    Std
Id  (Intercept)  0.946  0.973

No random effect correlations specified

Fixed effects:



C:\Users\jlpsc\anaconda3\envs\pymer4_projects\lib\site-packages\pymer4\models\Lmer.py:733: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ran_vars = ran_vars.applymap(


,Estimate,2.5_ci,97.5_ci,SE,OR,OR_2.5_ci,OR_97.5_ci,Prob,Prob_2.5_ci,Prob_97.5_ci,Z-stat,P-val,Sig
(Intercept),-2.305,-4.181,-0.428,0.958,0.100,0.015,0.652,0.091,0.015,0.395,-2.407,0.016,*
duty_momentmiddle,-2.437,-3.211,-1.663,0.395,0.087,0.040,0.189,0.080,0.039,0.159,-6.174,0.000,***
duty_momentstart,-5.234,-7.280,-3.188,1.044,0.005,0.001,0.041,0.005,0.001,0.040,-5.015,0.000,***
DMod_fatearly-start,-0.587,-1.604,0.429,0.518,0.556,0.201,1.535,0.357,0.167,0.606,-1.133,0.257,
DMod_fatni,1.037,0.308,1.767,0.372,2.822,1.360,5.854,0.738,0.576,0.854,2.786,0.005,**
Sleep_quality,-4.130,-5.720,-2.539,0.811,0.016,0.003,0.079,0.016,0.003,0.073,-5.089,0.000,***
time_awake,3.711,2.217,5.204,0.762,40.878,9.179,182.050,0.976,0.902,0.995,4.869,0.000,***
duty_length,3.405,1.787,5.023,0.826,30.120,5.971,151.924,0.968,0.857,0.993,4.124,0.000,***
agemiddle,1.206,0.293,2.118,0.465,3.339,1.341,8.313,0.770,0.573,0.893,2.590,0.010,**


As we can see above, for the first iteration, some covariates have no statistically significant effect on perception of sleepiness. However from exploratory data analysis when assessed isolated they present some effects on perception of sleepiness. So, the following transformations is proposed:

>- For **time_awake** we transform this covariate into an indicator of time awake before duty greater than 10 hours.
>
>- For **sleep duration** we transform this covariate in an indicator of sleep duration less than 5 hours.
>
>- For **sleep devaition**, we will not explore further this covariate for perception of sleepiness.

## 5.3. Second Iteration

### 5.3.1. Applying the transformations proposed

In [ ]:
df['sleep_dur'] = np.where(df['sleep_duration']< (5/(12.417-2.467)), 1, 0)
df['t_awake'] = np.where(df['time_awake'] > (10/(17.400000-0.250000)), 1, 0)

### 5.3.2. Running the second iteration

In [ ]:
covariates = ['TPFS_slp','duty_moment','DMod_slp','Sleep_quality','sleep_dur','t_awake']

formula = get_formula(covariates, 'kssd', prnt=False)
ff = formula + '+(1|Id)'

print('----------------------------------------------------------------------------------------------------------')
print(f'\033[91mFormula: {ff}\033[0m')
print('----------------------------------------------------------------------------------------------------------')

model_LME = Lmer(ff, data = df, family = 'binomial')
mdf = model_LME.fit()

mdf.head(100)

As we can see above, all covariates have pvalues bellow 0.05, meaning that all the covariates retained are statistically significant at the level of 0.05.

# 6. Model Assessment

## 6.1. Overdispersion Ratio

The Overdispersion ratio for binomial model is given by:

$$ OD = \frac{ \sum_{i=1}^{n} r_i^2}{df}$$

where $r_i^2$ is the squared sum of residuals and $df$ is the degrees of freedom of the model.

In [ ]:
res = model_LME.residuals
pearson = np.sum((res)**2)
df_resid = model_LME.data.shape[0] - len(model_LME.fixef)
overdispersion_ratio = pearson / df_resid

print('-------------------------------')
print(f'\033[91m OD = {round(overdispersion_ratio, 3)}\033[0m')
print('-------------------------------')